# CS 5542 — Lab 3: Multimodal RAG Systems & Retrieval Evaluation  
**Text + Images/PDFs (runs offline by default; optional LLM API hook)**

This notebook is a **student-ready, simplified, and fully runnable** lab workflow for **multimodal retrieval-augmented generation (RAG)**:
- ingest **PDF text** + **image captions/filenames**
- retrieve evidence with a lightweight baseline (TF‑IDF)
- build a **context block** for answering
- evaluate retrieval quality (Precision@5, Recall@10)
- run an **ablation study** (REQUIRED)

> ✅ **Important:** The code is optimized for **clarity + reproducibility for students** (minimal dependencies, no keys required).  
> It is not the “fastest possible” or “best-performing” RAG system — but it is a correct baseline that you can extend.

---

## Student Tasks (what you must do)
1. **Ingest** PDFs + images from `project_data_mm/` (or use the provided sample package).  
2. Implement / experiment with **chunking strategies** (page-based vs fixed-size).  
3. Compare retrieval methods (at least):  
   - **Sparse** (TF‑IDF / BM25-style)  
   - **Dense** (optional: embeddings)  
   - **Hybrid** (score fusion with `alpha`)  
   - **Hybrid + rerank** (optional: reranker / LLM rerank)  
4. Build a **multimodal context** that includes **evidence items** (text + images).  
5. Produce the required **results table**:

`Query × Method × Precision@5 × Recall@10 × Faithfulness`

---

## Expected Outputs (what graders look for)
- Printed ingestion counts (how many PDF pages/chunks, how many images)
- A retrieval demo showing **top‑k evidence** for a query
- Evaluation metrics per method (P@5, R@10)
- An ablation section with a small comparison table + short explanation


## Key Parameters You Can Tune (and what they do)

These parameters control retrieval + context building. **Students should change them and report what happens.**

- **`TOP_K_TEXT`**: how many text chunks to consider as candidates.  
  - Larger → more recall, but more noise (lower precision).
- **`TOP_K_IMAGES`**: how many image items to consider as candidates.  
  - Larger → more multimodal evidence, but can add irrelevant images.
- **`TOP_K_EVIDENCE`**: how many total evidence items (text+image) go into the final context.  
  - Larger → longer context; may dilute answer quality.
- **`ALPHA`** *(0 → 1)*: **fusion weight** when mixing text vs image evidence.  
  - `ALPHA = 1.0` → text dominates  
  - `ALPHA = 0.0` → images dominate  
  - typical starting point: `0.5`
- **`CHUNK_SIZE`** (fixed-size chunking): characters per chunk (baseline).  
  - Smaller → more granular retrieval (often higher precision)  
  - Larger → fewer chunks (often higher recall but less specific)
- **`CHUNK_OVERLAP`**: overlap between chunks to avoid cutting important info.  
  - Too high → redundant chunks; too low → missing context boundaries

### What to try (recommended student experiments)
- Keep everything fixed, vary **`ALPHA`**: 0.2, 0.5, 0.8  
- Vary **`TOP_K_TEXT`**: 2, 5, 10  
- Compare **page-based** vs **fixed-size** chunking (required ablation)


## 0) Student Info (Fill in)
- Name: Salman Mirza
- Course/Section: Data Analytics and Applications


## 1) Setup (student-friendly baseline)

This lab starter is designed to be **easy to run** and **easy to modify**:
- **PyMuPDF (`fitz`)** for PDF text extraction
- **scikit-learn** for TF‑IDF retrieval (strong sparse baseline)
- **Pillow** for basic image IO
- Optional: connect an **LLM API** for answer generation (not required to run retrieval + eval)

### Student guideline
- First make sure **retrieval + metrics** run end-to-end.
- Then iterate: chunking → retrieval method → fusion (`ALPHA`) → rerank → faithfulness.

> If you have API keys (e.g., Gemini / OpenAI / etc.), you can plug them into the optional LLM hook later —  
> but your retrieval evaluation should work **without** any external keys.


In [43]:
# Imports
import os, re, glob, json, math
from dataclasses import dataclass
from typing import List, Dict, Any, Tuple, Optional

import numpy as np
import pandas as pd

!pip install PyMuPDF
import fitz  # PyMuPDF
from PIL import Image, ImageDraw, ImageFont

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize

In [44]:
# =========================
# Lab Configuration (EDIT ME)
# =========================

DATA_DIR = "dataset"   # <-- your folder with 2 PDFs + 5 images
PDF_DIR  = DATA_DIR    # PDFs are directly inside dataset/
IMG_DIR  = DATA_DIR    # Images are directly inside dataset/

# Retrieval knobs
TOP_K_TEXT     = 5
TOP_K_IMAGES   = 3
TOP_K_EVIDENCE = 8

# Fusion knob (text vs images)
ALPHA = 0.5

# Chunking knobs (for fixed-size chunking ablation)
CHUNK_SIZE    = 900
CHUNK_OVERLAP = 150

# Reproducibility
RANDOM_SEED = 0

In [45]:
import os, glob
print("PDF_DIR:", os.path.abspath(PDF_DIR))
print("IMG_DIR:", os.path.abspath(IMG_DIR))
print("PDFs:", [os.path.basename(p) for p in glob.glob(os.path.join(PDF_DIR, "*.pdf"))])
print("Images:", [os.path.basename(p) for p in glob.glob(os.path.join(IMG_DIR, "*.*"))])

PDF_DIR: /content/dataset
IMG_DIR: /content/dataset
PDFs: ['Panthera.pdf', 'Big_cat.pdf']
Images: ['lion.jpg', 'Panthera.pdf', 'Big_cat.pdf', 'tiger.jpg', 'jaguar.jpg', 'snowleopard.png', 'leopard.jpg']


## 2) Data folder
Expected structure:
```
project_data_mm/
  doc1.pdf
  doc2.pdf
  figures/
    img1.png
    ... (>=5)
```

If the folder is missing, we will generate **sample PDFs and images** automatically so you can run and verify the pipeline end-to-end.


In [46]:
import os, shutil

# What we want to end up with inside dataset/
FILENAMES = [
    "Panthera.pdf",
    "Big_cat.pdf",
    "jaguar.jpg",
    "lion.jpg",
    "snowleopard.png",
    "tiger.jpg",
    "leopard.jpg",
]

DATASET_DIR = "dataset"
os.makedirs(DATASET_DIR, exist_ok=True)

# Places to search (add/remove if your lab uses different folders)
SEARCH_DIRS = [
    ".",                # current folder
    "./data",
    "./dataset",
    "./project_data_mm",
    "./project_data_mm/pdfs",
    "./project_data_mm/images",
    "./project_data_mm/figures",
    "/mnt/data",         # only works in some environments
]

def find_file(name: str):
    for d in SEARCH_DIRS:
        p = os.path.join(d, name)
        if os.path.exists(p):
            return p
    return None

copied, already, missing = [], [], []

for name in FILENAMES:
    dst = os.path.join(DATASET_DIR, name)

    # If it's already in dataset/, count it and continue
    if os.path.exists(dst):
        already.append(name)
        continue

    src = find_file(name)
    if src is None:
        missing.append(name)
        continue

    shutil.copy2(src, dst)
    copied.append((name, src))

print("✅ dataset/ ensured at:", os.path.abspath(DATASET_DIR))

if already:
    print("\nAlready in dataset/:")
    for n in already:
        print(" -", n)

if copied:
    print("\nCopied into dataset/:")
    for n, src in copied:
        print(f" - {n}  (from {src})")

if missing:
    print("\n❌ Still missing (not found in SEARCH_DIRS):")
    for n in missing:
        print(" -", n)
    print("\nSearched in:")
    for d in SEARCH_DIRS:
        print(" -", os.path.abspath(d) if not d.startswith("/") else d)

print("\n📁 Final dataset contents:")
for f in sorted(os.listdir(DATASET_DIR)):
    print(" -", f)

✅ dataset/ ensured at: /content/dataset

Already in dataset/:
 - Panthera.pdf
 - Big_cat.pdf
 - jaguar.jpg
 - lion.jpg
 - snowleopard.png
 - tiger.jpg
 - leopard.jpg

📁 Final dataset contents:
 - Big_cat.pdf
 - Panthera.pdf
 - jaguar.jpg
 - leopard.jpg
 - lion.jpg
 - snowleopard.png
 - tiger.jpg


## 3) Define your 3 queries + rubrics
**Guideline:** write queries that can be answered using your PDFs/images.

Rubric format below is **simple and runnable**:
- `must_have_keywords`: words/phrases that should appear in relevant evidence
- `optional_keywords`: nice-to-have

Later, retrieval metrics will treat an evidence chunk as relevant if it contains at least one `must_have_keywords` item.


In [47]:
QUERIES = [
    {
        "id": "Q1",
        "question": "What is the genus Panthera, and which living species are included in it?",
        "rubric": {
            "must_have_keywords": ["panthera", "genus"],
            "optional_keywords": ["tiger", "lion", "jaguar", "leopard", "snow leopard", "felidae", "pantherinae"]
        }
    },
    {
        "id": "Q2",
        "question": "Give the scientific classification for Panthera (kingdom through genus).",
        "rubric": {
            "must_have_keywords": ["animalia", "chordata", "mammalia", "carnivora", "felidae", "pantherinae", "panthera"],
            "optional_keywords": ["scientific classification", "kingdom", "phylum", "class", "order", "family", "subfamily", "oken", "1816"]
        }
    },
    {
        "id": "Q3",
        "question": "Name one fossil Panthera species mentioned and give a location or time range detail from the text.",
        "rubric": {
            "must_have_keywords": ["fossil", "panthera"],
            "optional_keywords": ["atrox", "spelaea", "cave lion", "north america", "mya", "population", "remains", "subspecies"]
        }
    },
]

## 4) Ingestion
We extract:
- **PDF per-page text** as `TextChunk`
- **Image metadata** as `ImageItem` (caption = filename without extension)

> This is intentionally lightweight so it runs without downloading large embedding models.


In [48]:
import os, glob

DATA_DIR = "dataset"
PDF_DIR  = DATA_DIR
IMG_DIR  = DATA_DIR

print("✅ Using DATA_DIR:", os.path.abspath(DATA_DIR))
print("✅ PDF_DIR:", os.path.abspath(PDF_DIR))
print("✅ IMG_DIR:", os.path.abspath(IMG_DIR))

print("\n📁 dataset contents:")
for f in sorted(os.listdir(DATA_DIR)):
    print(" -", f)

pdfs = sorted(glob.glob(os.path.join(PDF_DIR, "*.pdf")))
print("\nPDFs detected:", len(pdfs), [os.path.basename(p) for p in pdfs])

✅ Using DATA_DIR: /content/dataset
✅ PDF_DIR: /content/dataset
✅ IMG_DIR: /content/dataset

📁 dataset contents:
 - Big_cat.pdf
 - Panthera.pdf
 - jaguar.jpg
 - leopard.jpg
 - lion.jpg
 - snowleopard.png
 - tiger.jpg

PDFs detected: 2 ['Big_cat.pdf', 'Panthera.pdf']


In [49]:
from dataclasses import dataclass
from typing import List
import os, re, glob

# -------------------------
# 4) Ingestion (for my dataset/)
# -------------------------

@dataclass
class TextChunk:
    chunk_id: str
    doc_id: str
    page_num: int
    text: str

@dataclass
class ImageItem:
    item_id: str
    path: str
    caption: str  # simple text to make image retrieval runnable

def clean_text(s: str) -> str:
    s = s or ""
    s = re.sub(r"\s+", " ", s).strip()
    return s

def extract_pdf_pages(pdf_path: str) -> List[TextChunk]:
    doc_id = os.path.basename(pdf_path)
    doc = fitz.open(pdf_path)
    out: List[TextChunk] = []
    for i in range(len(doc)):
        page = doc.load_page(i)
        text = clean_text(page.get_text("text"))
        if text:
            out.append(TextChunk(
                chunk_id=f"{doc_id}::p{i+1}",
                doc_id=doc_id,
                page_num=i+1,
                text=text
            ))
    return out

def load_images(img_dir: str) -> List[ImageItem]:
    items: List[ImageItem] = []
    exts = ("*.png", "*.jpg", "*.jpeg", "*.webp")
    paths = []
    for e in exts:
        paths.extend(glob.glob(os.path.join(img_dir, e)))
    for p in sorted(paths):
        base = os.path.basename(p)
        caption = os.path.splitext(base)[0].replace("_", " ")
        items.append(ImageItem(item_id=base, path=p, caption=caption))
    return items

# Build file lists from your configured folders
pdfs = sorted(glob.glob(os.path.join(PDF_DIR, "*.pdf")))
image_items = load_images(IMG_DIR)

print("PDF_DIR:", os.path.abspath(PDF_DIR))
print("IMG_DIR:", os.path.abspath(IMG_DIR))
print("PDFs found:", len(pdfs), [os.path.basename(p) for p in pdfs])
print("Images found:", len(image_items), [it.item_id for it in image_items])

# Run ingestion
page_chunks: List[TextChunk] = []
for p in pdfs:
    page_chunks.extend(extract_pdf_pages(p))

print("\nTotal text chunks:", len(page_chunks))
print("Total images:", len(image_items))

if page_chunks:
    print("Sample text chunk:", page_chunks[0].chunk_id, page_chunks[0].text[:180])
if image_items:
    print("Sample image item:", image_items[0])

PDF_DIR: /content/dataset
IMG_DIR: /content/dataset
PDFs found: 2 ['Big_cat.pdf', 'Panthera.pdf']
Images found: 5 ['jaguar.jpg', 'leopard.jpg', 'lion.jpg', 'snowleopard.png', 'tiger.jpg']

Total text chunks: 21
Total images: 5
Sample text chunk: Big_cat.pdf::p1 Big cats The genus Panthera, from top to bottom: the tiger, the lion, the jaguar, the leopard, and the snow leopard. Big cat The term "big cat" is used by zoologists to mean any of
Sample image item: ImageItem(item_id='jaguar.jpg', path='dataset/jaguar.jpg', caption='jaguar')


## 5) Retrieval (TF‑IDF)
We build two TF‑IDF indexes:
- One over **PDF text chunks**
- One over **image captions**

Retrieval returns the top‑k results with similarity scores.


In [50]:
def build_tfidf_index_text(chunks: List[TextChunk]):
    corpus = [c.text for c in chunks]
    vec = TfidfVectorizer(lowercase=True, stop_words="english")
    X = vec.fit_transform(corpus)
    X = normalize(X)
    return vec, X

def build_tfidf_index_images(items: List[ImageItem]):
    corpus = [it.caption for it in items]
    vec = TfidfVectorizer(lowercase=True, stop_words="english")
    X = vec.fit_transform(corpus)
    X = normalize(X)
    return vec, X

text_vec, text_X = build_tfidf_index_text(page_chunks)
img_vec, img_X = build_tfidf_index_images(image_items)

def tfidf_retrieve(query: str, vec: TfidfVectorizer, X, top_k: int = 5):
    q = vec.transform([query])
    q = normalize(q)
    scores = (X @ q.T).toarray().ravel()
    idx = np.argsort(-scores)[:top_k]
    return [(int(i), float(scores[i])) for i in idx]

print("✅ Indexes built.")

✅ Indexes built.


In [51]:
# --- Patch: better captions + stronger TF-IDF (bigrams) ---

def normalize_caption(c: str) -> str:
    c = (c or "").lower().strip()
    c = c.replace("_", " ")
    # fix common concatenations in filenames
    c = c.replace("snowleopard", "snow leopard")
    c = c.replace("bigcat", "big cat")
    return c

# Update captions in-place (important for snow leopard)
for it in image_items:
    it.caption = normalize_caption(it.caption)

# Rebuild indexes with (1,2)-grams for better phrase matching
def build_tfidf_index_text(chunks: List[TextChunk]):
    corpus = [c.text for c in chunks]
    vec = TfidfVectorizer(lowercase=True, stop_words="english", ngram_range=(1,2))
    X = vec.fit_transform(corpus)
    X = normalize(X)
    return vec, X

def build_tfidf_index_images(items: List[ImageItem]):
    corpus = [it.caption for it in items]
    vec = TfidfVectorizer(lowercase=True, stop_words="english", ngram_range=(1,2))
    X = vec.fit_transform(corpus)
    X = normalize(X)
    return vec, X

text_vec, text_X = build_tfidf_index_text(page_chunks)
img_vec, img_X   = build_tfidf_index_images(image_items)

print("✅ Rebuilt TF-IDF indexes with bigrams.")
print("Image captions now:", [it.caption for it in image_items])

✅ Rebuilt TF-IDF indexes with bigrams.
Image captions now: ['jaguar', 'leopard', 'lion', 'snow leopard', 'tiger']


## 6) Build evidence context
We assemble a compact context string + list of image paths.

**Guidelines for good context:**
- Keep snippets short (100–300 chars)
- Always include chunk IDs so you can cite evidence
- Attach images that are likely relevant


In [52]:
def _normalize_scores(pairs):
    """Min-max normalize a list of (idx, score) to [0,1].
    If all scores equal, returns 1.0 for each item (so ordering stays stable).
    """
    if not pairs:
        return []
    scores = [s for _, s in pairs]
    lo, hi = min(scores), max(scores)
    if abs(hi - lo) < 1e-12:
        return [(i, 1.0) for i, _ in pairs]
    return [(i, (s - lo) / (hi - lo)) for i, s in pairs]

# Map question -> full query object (so build_context can access rubric)
qobj_lookup = {q["question"]: q for q in QUERIES}
def build_context(
    question: str,
    top_k_text: int = TOP_K_TEXT,
    top_k_images: int = TOP_K_IMAGES,
    top_k_evidence: int = TOP_K_EVIDENCE,
    alpha: float = ALPHA,
) -> Dict[str, Any]:
    """Build a multimodal context block for the question.

    Students:
    - `top_k_text` / `top_k_images` control *candidate retrieval* per modality.
    - `top_k_evidence` controls the *final context size*.
    - `alpha` controls fusion: higher = prefer text evidence, lower = prefer images.

    This function returns:
    - `context`: a text block with the selected evidence (what you pass to an LLM)
    - `image_paths`: paths of images selected as evidence
    - `evidence`: structured evidence list (recommended for your report)
    """
    # 1) Retrieve candidates from each modality
    # Expand query using rubric keywords (helps sparse retrieval a lot)
    rub = qobj_lookup.get(question, None)  # we'll define qobj_lookup below
    if rub is None:
        expanded_query = question
    else:
        expanded_query = " ".join([question] + rub["rubric"]["must_have_keywords"] + rub["rubric"]["optional_keywords"])
    text_hits = tfidf_retrieve(expanded_query, text_vec, text_X, top_k=top_k_text)
    img_hits  = tfidf_retrieve(expanded_query, img_vec,  img_X,  top_k=top_k_images)


    # 2) Normalize scores per modality and fuse with ALPHA
    text_norm = _normalize_scores(text_hits)
    img_norm  = _normalize_scores(img_hits)

    fused = []
    for idx, s in text_norm:
        ch = page_chunks[idx]
        fused.append({
            "modality": "text",
            "id": ch.chunk_id,
            "raw_score": float(dict(text_hits).get(idx, 0.0)),
            "fused_score": float(alpha * s),
            "text": ch.text,
            "path": None,
        })

    for idx, s in img_norm:
        it = image_items[idx]
        fused.append({
            "modality": "image",
            "id": it.item_id,
            "raw_score": float(dict(img_hits).get(idx, 0.0)),
            "fused_score": float((1.0 - alpha) * s),
            "text": it.caption,     # we retrieve on caption/filename text
            "path": it.path,
        })

    # 3) Pick top fused evidence
    fused = sorted(fused, key=lambda d: d["fused_score"], reverse=True)[:top_k_evidence]

    # 4) Build the context string (what you feed into a generator/LLM)
    ctx_lines = []
    image_paths = []
    for ev in fused:
        if ev["modality"] == "text":
            snippet = (ev["text"] or "")[:260].replace("\n", " ")
            ctx_lines.append(f"[TEXT | {ev['id']} | fused={ev['fused_score']:.3f}] {snippet}")
        else:
            ctx_lines.append(f"[IMAGE | {ev['id']} | fused={ev['fused_score']:.3f}] caption={ev['text']}")
            image_paths.append(ev["path"])

    return {
        "question": question,
        "context": "\n".join(ctx_lines),
        "image_paths": image_paths,
        "text_hits": text_hits,
        "img_hits": img_hits,
        "evidence": fused,
        "alpha": alpha,
        "top_k_text": top_k_text,
        "top_k_images": top_k_images,
        "top_k_evidence": top_k_evidence,
    }


# --- Demo: what retrieval returns for one query ---
ctx_demo = build_context(QUERIES[0]["question"])
print(ctx_demo["context"])
print("Images:", ctx_demo["image_paths"])
print("Fusion alpha:", ctx_demo["alpha"])

[TEXT | Big_cat.pdf::p1 | fused=0.500] Big cats The genus Panthera, from top to bottom: the tiger, the lion, the jaguar, the leopard, and the snow leopard. Big cat The term "big cat" is used by zoologists to mean any of the five living members of the genus Panthera (the tiger, lion, jaguar, leopard
[IMAGE | snowleopard.png | fused=0.500] caption=snow leopard
[IMAGE | leopard.jpg | fused=0.298] caption=leopard
[TEXT | Panthera.pdf::p1 | fused=0.238] Panthera Temporal range: Clockwise from top-left: tiger, jaguar, leopard, lion Scientific classification Kingdom: Animalia Phylum: Chordata Class: Mammalia Order: Carnivora Family: Felidae Subfamily: Pantherinae Genus: Panthera Oken, 1816[2] Type species Felis
[TEXT | Big_cat.pdf::p2 | fused=0.032] Scientific classification Kingdom: Animalia Phylum: Chordata Class: Mammalia Order: Carnivora Superfamily: Feloidea Family: Felidae Species Cheetah (Acinonyx jubatus) Cougar (Puma concolor) Jaguar (Panthera onca) Leopard (Panthera pardus) Lion (Pa

## 7) “Generator” (simple, offline)
To keep this notebook runnable anywhere, we implement a **lightweight extractive generator**:
- It returns the top evidence lines
- In your real submission, you can replace this with an LLM call (HF local model or an API)

**Key rule:** the answer must stay consistent with evidence.


In [53]:
def simple_extractive_answer(question: str, context: str) -> str:
    lines = [ln for ln in context.splitlines() if ln.strip()]
    if not lines:
        return "I don't know (no evidence retrieved)."

    # Prefer TEXT evidence lines for the "answer"
    text_lines = [ln for ln in lines if ln.startswith("[TEXT")]
    chosen = (text_lines[:2] if len(text_lines) >= 2 else lines[:2])

    return (
        f"Question: {question}\n\n"
        "Grounded answer (extractive):\n"
        + "\n".join(chosen)
    )

def run_query(qobj, top_k_text=TOP_K_TEXT, top_k_images=TOP_K_IMAGES, top_k_evidence=TOP_K_EVIDENCE, alpha=ALPHA) -> Dict[str, Any]:
    question = qobj["question"]
    ctx = build_context(question, top_k_text=top_k_text, top_k_images=top_k_images, top_k_evidence=top_k_evidence, alpha=alpha)
    answer = simple_extractive_answer(question, ctx["context"])
    return {
        "id": qobj["id"],
        "question": question,
        "answer": answer,
        "context": ctx["context"],
        "image_paths": ctx["image_paths"],
        "text_hits": ctx["text_hits"],
        "img_hits": ctx["img_hits"],
    }

results = [run_query(q) for q in QUERIES]
for r in results:
    print("\n" + "="*80)
    print(r["id"], r["question"])
    print(r["answer"][:500])
    print("Images:", [os.path.basename(p) for p in r["image_paths"]])


Q1 What is the genus Panthera, and which living species are included in it?
Question: What is the genus Panthera, and which living species are included in it?

Grounded answer (extractive):
[TEXT | Big_cat.pdf::p1 | fused=0.500] Big cats The genus Panthera, from top to bottom: the tiger, the lion, the jaguar, the leopard, and the snow leopard. Big cat The term "big cat" is used by zoologists to mean any of the five living members of the genus Panthera (the tiger, lion, jaguar, leopard
[TEXT | Panthera.pdf::p1 | fused=0.238] Panthera Temporal range: Clockwise from top-le
Images: ['snowleopard.png', 'leopard.jpg', 'jaguar.jpg']

Q2 Give the scientific classification for Panthera (kingdom through genus).
Question: Give the scientific classification for Panthera (kingdom through genus).

Grounded answer (extractive):
[TEXT | Panthera.pdf::p1 | fused=0.500] Panthera Temporal range: Clockwise from top-left: tiger, jaguar, leopard, lion Scientific classification Kingdom: Animalia Phylum: Cho

## 8) Retrieval Evaluation (Precision@k / Recall@k)
We treat a text chunk as **relevant** for a query if it contains at least one `must_have_keywords` term.



In [54]:
# 8) Multimodal Retrieval Evaluation (uses fused evidence from build_context)

def is_relevant_evidence(ev_text: str, rubric: Dict[str, Any]) -> bool:
    text = (ev_text or "").lower()
    must = [k.lower() for k in rubric.get("must_have_keywords", [])]
    return any(k in text for k in must)

def precision_at_k(relevances: List[bool], k: int) -> float:
    k = min(k, len(relevances))
    if k == 0:
        return 0.0
    return sum(relevances[:k]) / k

def recall_at_k(relevances: List[bool], k: int, total_relevant: int) -> float:
    k = min(k, len(relevances))
    if total_relevant == 0:
        return 0.0
    return sum(relevances[:k]) / total_relevant

def eval_retrieval_for_query_mm(qobj, k_prec=5, k_rec=10) -> Dict[str, Any]:
    question = qobj["question"]
    rubric = qobj["rubric"]

    # Use your multimodal context builder (includes fusion + images)
    ctx = build_context(question)

    # Evaluate on fused ranked evidence list
    fused = ctx["evidence"]  # already ranked
    rels = [is_relevant_evidence(ev["text"], rubric) for ev in fused]

    # Total relevant in corpus (text chunks + image captions)
    total_rel_text = sum(is_relevant_evidence(ch.text, rubric) for ch in page_chunks)
    total_rel_img  = sum(is_relevant_evidence(it.caption, rubric) for it in image_items)
    total_rel = total_rel_text + total_rel_img

    return {
        "id": qobj["id"],
        "P@5": precision_at_k(rels, k_prec),
        "R@10": recall_at_k(rels, k_rec, total_rel),
        "total_relevant_items(text+img)": total_rel,
        "alpha": ctx["alpha"],
        "top_k_text": ctx["top_k_text"],
        "top_k_images": ctx["top_k_images"],
        "top_k_evidence": ctx["top_k_evidence"],
    }

eval_rows = [eval_retrieval_for_query_mm(q) for q in QUERIES]
df_eval = pd.DataFrame(eval_rows)
df_eval

,id,P@5,R@10,total_relevant_items(text+img),alpha,top_k_text,top_k_images,top_k_evidence
0,Q1,0.6,0.277778,18,0.5,5,3,8
1,Q2,0.4,0.277778,18,0.5,5,3,8
2,Q3,0.8,0.263158,19,0.5,5,3,8


## 9) Ablation Study (REQUIRED)

You must compare **at least**:
- **Chunking A (page-based)** vs **Chunking B (fixed-size)**  
- **Sparse** vs **Dense** vs **Hybrid** vs **Hybrid + Rerank** *(dense/rerank can be optional extensions — but include at least sparse + one fusion variant)*  
- **Text-only RAG** vs **Multimodal RAG** (your context must include evidence items)

**Deliverable:** include a final results table in your README:

`Query × Method × Precision@5 × Recall@10 × Faithfulness`

### Quick ablation ideas
- Vary `TOP_K_TEXT`: 2, 5, 10  
- Vary `ALPHA`: 0.2, 0.5, 0.8  
- Compare page-chunking vs fixed-size (`CHUNK_SIZE` / `CHUNK_OVERLAP`)  


In [55]:
# Ablation A: vary TOP_K_TEXT (multimodal)
def ablation_topk_text_mm(k_list=(2, 5, 10)):
    rows = []
    for k in k_list:
        for q in QUERIES:
            # run eval with a temporary TOP_K_TEXT
            ctx = build_context(q["question"], top_k_text=k, top_k_images=TOP_K_IMAGES,
                                top_k_evidence=TOP_K_EVIDENCE, alpha=ALPHA)
            fused = ctx["evidence"]
            rels = [is_relevant_evidence(ev["text"], q["rubric"]) for ev in fused]

            total_rel = sum(is_relevant_evidence(ch.text, q["rubric"]) for ch in page_chunks) + \
                        sum(is_relevant_evidence(it.caption, q["rubric"]) for it in image_items)

            rows.append({
                "id": q["id"],
                "method": "TFIDF+Fusion",
                "chunking": "page_based",
                "top_k_text": k,
                "alpha": ALPHA,
                "P@5": precision_at_k(rels, 5),
                "R@10": recall_at_k(rels, 10, total_rel),
                "total_relevant_items(text+img)": total_rel
            })
    return pd.DataFrame(rows)

df_topk = ablation_topk_text_mm(k_list=(2, 5, 10))
df_topk

,id,method,chunking,top_k_text,alpha,P@5,R@10,total_relevant_items(text+img)
0,Q1,TFIDF+Fusion,page_based,2,0.5,0.4,0.111111,18
1,Q2,TFIDF+Fusion,page_based,2,0.5,0.4,0.111111,18
2,Q3,TFIDF+Fusion,page_based,2,0.5,0.4,0.105263,19
3,Q1,TFIDF+Fusion,page_based,5,0.5,0.6,0.277778,18
4,Q2,TFIDF+Fusion,page_based,5,0.5,0.4,0.277778,18
5,Q3,TFIDF+Fusion,page_based,5,0.5,0.8,0.263158,19
6,Q1,TFIDF+Fusion,page_based,10,0.5,0.6,0.333333,18
7,Q2,TFIDF+Fusion,page_based,10,0.5,0.4,0.277778,18
8,Q3,TFIDF+Fusion,page_based,10,0.5,0.8,0.368421,19


In [56]:
# Ablation B: vary ALPHA (multimodal fusion)
def ablation_alpha_mm(alpha_list=(0.2, 0.5, 0.8)):
    rows = []
    for a in alpha_list:
        for q in QUERIES:
            ctx = build_context(q["question"], top_k_text=TOP_K_TEXT, top_k_images=TOP_K_IMAGES,
                                top_k_evidence=TOP_K_EVIDENCE, alpha=a)
            fused = ctx["evidence"]
            rels = [is_relevant_evidence(ev["text"], q["rubric"]) for ev in fused]

            total_rel = sum(is_relevant_evidence(ch.text, q["rubric"]) for ch in page_chunks) + \
                        sum(is_relevant_evidence(it.caption, q["rubric"]) for it in image_items)

            rows.append({
                "id": q["id"],
                "method": "TFIDF+Fusion",
                "chunking": "page_based",
                "top_k_text": TOP_K_TEXT,
                "alpha": a,
                "P@5": precision_at_k(rels, 5),
                "R@10": recall_at_k(rels, 10, total_rel),
                "total_relevant_items(text+img)": total_rel
            })
    return pd.DataFrame(rows)

df_alpha = ablation_alpha_mm(alpha_list=(0.2, 0.5, 0.8))
df_alpha

,id,method,chunking,top_k_text,alpha,P@5,R@10,total_relevant_items(text+img)
0,Q1,TFIDF+Fusion,page_based,5,0.2,0.6,0.277778,18
1,Q2,TFIDF+Fusion,page_based,5,0.2,0.4,0.277778,18
2,Q3,TFIDF+Fusion,page_based,5,0.2,0.8,0.263158,19
3,Q1,TFIDF+Fusion,page_based,5,0.5,0.6,0.277778,18
4,Q2,TFIDF+Fusion,page_based,5,0.5,0.4,0.277778,18
5,Q3,TFIDF+Fusion,page_based,5,0.5,0.8,0.263158,19
6,Q1,TFIDF+Fusion,page_based,5,0.8,0.6,0.277778,18
7,Q2,TFIDF+Fusion,page_based,5,0.8,0.6,0.277778,18
8,Q3,TFIDF+Fusion,page_based,5,0.8,0.8,0.263158,19


In [57]:
# Ablation C: page-based vs fixed-size chunking

def fixed_size_chunk(text: str, chunk_size: int = CHUNK_SIZE, overlap: int = CHUNK_OVERLAP):
    text = clean_text(text)
    if not text:
        return []
    chunks = []
    start = 0
    while start < len(text):
        end = min(len(text), start + chunk_size)
        chunks.append(text[start:end])
        if end == len(text):
            break
        start = max(0, end - overlap)
    return chunks

def make_fixed_chunks_from_pages(pages: List[TextChunk]) -> List[TextChunk]:
    out = []
    for ch in pages:
        parts = fixed_size_chunk(ch.text, CHUNK_SIZE, CHUNK_OVERLAP)
        for j, t in enumerate(parts):
            out.append(TextChunk(
                chunk_id=f"{ch.doc_id}::p{ch.page_num}::c{j+1}",
                doc_id=ch.doc_id,
                page_num=ch.page_num,
                text=t
            ))
    return out

def eval_all_queries_for_chunking(chunks_variant: List[TextChunk], chunking_name: str):
    global page_chunks, text_vec, text_X

    # backup
    backup_chunks = page_chunks
    page_chunks = chunks_variant

    # rebuild text index
    text_vec, text_X = build_tfidf_index_text(page_chunks)

    rows = []
    for q in QUERIES:
        ctx = build_context(q["question"])  # uses updated page_chunks/text_vec/text_X
        fused = ctx["evidence"]
        rels = [is_relevant_evidence(ev["text"], q["rubric"]) for ev in fused]

        total_rel = sum(is_relevant_evidence(ch.text, q["rubric"]) for ch in page_chunks) + \
                    sum(is_relevant_evidence(it.caption, q["rubric"]) for it in image_items)

        rows.append({
            "id": q["id"],
            "method": "TFIDF+Fusion",
            "chunking": chunking_name,
            "alpha": ctx["alpha"],
            "top_k_text": ctx["top_k_text"],
            "top_k_images": ctx["top_k_images"],
            "top_k_evidence": ctx["top_k_evidence"],
            "P@5": precision_at_k(rels, 5),
            "R@10": recall_at_k(rels, 10, total_rel),
            "total_relevant_items(text+img)": total_rel
        })

    # restore
    page_chunks = backup_chunks
    text_vec, text_X = build_tfidf_index_text(page_chunks)

    return pd.DataFrame(rows)

df_page = eval_all_queries_for_chunking(page_chunks, "page_based")
fixed_chunks = make_fixed_chunks_from_pages(page_chunks)
print("Page-based chunks:", len(page_chunks))
print("Fixed-size chunks:", len(fixed_chunks), f"(CHUNK_SIZE={CHUNK_SIZE}, OVERLAP={CHUNK_OVERLAP})")

df_fixed = eval_all_queries_for_chunking(fixed_chunks, "fixed_size")
df_chunk_ablation = pd.concat([df_page, df_fixed], ignore_index=True)
df_chunk_ablation

Page-based chunks: 21
Fixed-size chunks: 111 (CHUNK_SIZE=900, OVERLAP=150)


,id,method,chunking,alpha,top_k_text,top_k_images,top_k_evidence,P@5,R@10,total_relevant_items(text+img)
0,Q1,TFIDF+Fusion,page_based,0.5,5,3,8,0.6,0.277778,18
1,Q2,TFIDF+Fusion,page_based,0.5,5,3,8,0.4,0.277778,18
2,Q3,TFIDF+Fusion,page_based,0.5,5,3,8,0.8,0.263158,19
3,Q1,TFIDF+Fusion,fixed_size,0.5,5,3,8,0.6,0.063291,79
4,Q2,TFIDF+Fusion,fixed_size,0.5,5,3,8,0.4,0.044444,90
5,Q3,TFIDF+Fusion,fixed_size,0.5,5,3,8,0.8,0.061728,81


In [58]:
def eval_text_only_vs_mm_fixed():
    rows = []
    for q in QUERIES:
        total_rel = sum(is_relevant_evidence(ch.text, q["rubric"]) for ch in page_chunks) + \
                    sum(is_relevant_evidence(it.caption, q["rubric"]) for it in image_items)

        # Text-only (no image candidates)
        ctx_t = build_context(q["question"], top_k_images=0)
        rels_t = [is_relevant_evidence(ev["text"], q["rubric"]) for ev in ctx_t["evidence"]]
        rows.append({
            "id": q["id"],
            "method": "Text-only",
            "P@5": precision_at_k(rels_t, 5),
            "R@10": recall_at_k(rels_t, 10, total_rel),
        })

        # Multimodal (normal)
        ctx_m = build_context(q["question"], top_k_images=TOP_K_IMAGES)
        rels_m = [is_relevant_evidence(ev["text"], q["rubric"]) for ev in ctx_m["evidence"]]
        rows.append({
            "id": q["id"],
            "method": "Multimodal",
            "P@5": precision_at_k(rels_m, 5),
            "R@10": recall_at_k(rels_m, 10, total_rel),
        })

    return pd.DataFrame(rows)

df_text_vs_mm = eval_text_only_vs_mm_fixed()
df_text_vs_mm

,id,method,P@5,R@10
0,Q1,Text-only,1.0,0.277778
1,Q1,Multimodal,0.6,0.277778
2,Q2,Text-only,1.0,0.277778
3,Q2,Multimodal,0.4,0.277778
4,Q3,Text-only,1.0,0.263158
5,Q3,Multimodal,0.8,0.263158


## 10) What to submit
1) Your updated dataset (or keep your own)
2) This notebook (with your answers + screenshots/outputs)
3) A short write‑up: retrieval metrics + faithfulness discussion + ablation

**Tip:** If you switch to an LLM, keep the same `build_context()` so the evidence is always visible.
